# Fundamentals of Machine Learning — Exercise 9

## Hyperparameter Optimisation with Optuna

A model can be implemented correctly and still perform poorly because its **hyperparameters** are unsuitable.

In this exercise we will use **Support Vector Machines (SVMs)** as the main tuning example. This connects the practical exercise on Optuna with the SVM topic from the lecture.

The central question is not:

> “How do I make Optuna find the largest score?”

It is:

> **How do I design, run, inspect, and defend a hyperparameter search without leaking information from the test set?**

The classroom workflow remains:

> **question → prediction → computation → visualisation → interpretation → verification**

### Learning outcomes

After completing this exercise, you should be able to:

- distinguish model parameters from hyperparameters,
- explain why hyperparameter tuning needs validation or cross-validation,
- keep a test set untouched until the final model decision,
- explain the roles of `C`, `gamma`, and `kernel` in an SVM,
- recognise when a hyperparameter should be searched on a logarithmic scale,
- define an Optuna `objective`, `trial`, and `study`,
- build conditional search spaces,
- inspect trial history instead of accepting `best_params` blindly,
- recognise a search boundary that may be too narrow,
- compare random search with Optuna's TPE sampler,
- use parameter importance cautiously,
- refit the selected configuration and perform one final test audit,
- use AI to challenge a search-space decision rather than generate it.

**Recommended duration:** 100–120 minutes.  
The final tuning memo may be completed as homework.

## How to work in this exercise

Most routine code is prepared.

Your task is to:

1. **predict** what changing a hyperparameter should do,
2. **justify** a search range before running Optuna,
3. **inspect** more than the single best trial,
4. **compare** validation evidence fairly,
5. **verify** that the test set stayed untouched,
6. **state** one limitation of the search.

A tuning run is an experiment.  
A larger number in one study is not automatically a better scientific conclusion.

## 0. Imports

In [ ]:
# If Optuna is missing in Google Colab, uncomment:
# !pip install -q optuna

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from optuna.samplers import TPESampler, RandomSampler
from optuna.importance import get_param_importances

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    recall_score,
    confusion_matrix,
)

sns.set_theme(style="whitegrid", context="notebook")
optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42

## 1. Parameters and hyperparameters are not the same thing

Consider an SVM classifier.

### Model parameters
These are learned from the training data.

Examples include the support vectors and quantities that define the fitted decision boundary.

### Hyperparameters
These are chosen **before or around training** and control how learning behaves.

For an SVM, important examples are:

- `kernel`,
- `C`,
- `gamma` for kernels such as RBF.

### Activity 1 — classify the choices

For each item, write **parameter**, **hyperparameter**, or **evaluation choice**:

| Item | Type |
|---|---|
| `C=10` | |
| fitted support vectors | |
| RBF kernel | |
| ROC-AUC as the primary metric | |
| decision function learned from training data | |
| 5-fold cross-validation | |

Why should Optuna optimise hyperparameters rather than learned model parameters?

## 2. Prepare the data and protect the test set

We use the Breast Cancer Wisconsin dataset.

We will create:

- a **development set** used for training and cross-validation,
- a **test set** used only once after model selection.

The test set must not influence:

- the search space,
- the best trial,
- the metric choice,
- the decision to rerun the study.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Development:", X_dev.shape)
print("Test:", X_test.shape)
print("Positive-class proportion, development:", round(y_dev.mean(), 3))
print("Positive-class proportion, test:", round(y_test.mean(), 3))

### Activity 2 — predict the leakage risk

Which of these actions are acceptable **before final model selection**?

- [ ] compute cross-validation ROC-AUC on the development set,
- [ ] try a wider `C` range because the best trial lies at the current boundary,
- [ ] inspect test ROC-AUC after every ten Optuna trials,
- [ ] choose between accuracy and ROC-AUC after seeing which looks better on the test set,
- [ ] compare two samplers using only development-set cross-validation.

Explain one unacceptable action in your own words.

## 3. SVM hyperparameters before Optuna

For an RBF SVM:

### `C`
Controls the penalty for training errors.

- smaller `C`: stronger regularisation, smoother / simpler boundary,
- larger `C`: stronger pressure to fit training observations.

### `gamma`
Controls how local the influence of each observation is in the RBF kernel.

- smaller `gamma`: broader influence, smoother boundary,
- larger `gamma`: highly local influence, more flexible boundary.

Both often vary meaningfully across **orders of magnitude**, which is why logarithmic search is useful.

Before running the experiment, predict:

1. What could happen when both `C` and `gamma` are very small?
2. What could happen when both are very large?
3. Why is searching `C` only in `[1, 2, 3, 4, 5]` a strange default?

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

manual_settings = [
    {"C": 0.01, "gamma": 0.001},
    {"C": 1.0,  "gamma": 0.01},
    {"C": 10.0, "gamma": 0.01},
    {"C": 100.0, "gamma": 0.1},
    {"C": 1000.0, "gamma": 1.0},
]

manual_rows = []

for setting in manual_settings:
    model = Pipeline([
        ("scale", StandardScaler()),
        ("svc", SVC(
            kernel="rbf",
            C=setting["C"],
            gamma=setting["gamma"],
        )),
    ])

    scores = cross_validate(
        model,
        X_dev,
        y_dev,
        cv=cv,
        scoring="roc_auc",
        return_train_score=True,
    )

    manual_rows.append({
        **setting,
        "train ROC-AUC": scores["train_score"].mean(),
        "validation ROC-AUC": scores["test_score"].mean(),
        "validation std": scores["test_score"].std(),
    })

manual_results = pd.DataFrame(manual_rows)
manual_results.round(4)

In [ ]:
plot_manual = manual_results.copy()
plot_manual["setting"] = [
    f"C={c:g}, gamma={g:g}"
    for c, g in zip(plot_manual["C"], plot_manual["gamma"])
]

plot_manual.set_index("setting")[
    ["train ROC-AUC", "validation ROC-AUC"]
].plot(
    kind="bar",
    figsize=(10, 4.5),
)

plt.ylim(0.5, 1.02)
plt.ylabel("ROC-AUC")
plt.title("A few manually chosen SVM configurations")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### Activity 3 — diagnose before automating

1. Which manual configuration looks too weak?
2. Is there a configuration with a larger train–validation gap?
3. Does the largest `C` produce the best validation score?
4. Why would testing hundreds of manually chosen combinations become inefficient?

Do not start Optuna until you can explain what the two main hyperparameters do.

## 4. What Optuna actually does

Four objects are enough to start:

- **trial** — one proposed hyperparameter configuration,
- **objective** — function that evaluates one trial,
- **study** — collection of trials,
- **sampler** — strategy that proposes new configurations.

The objective below returns the **mean cross-validation ROC-AUC**.

Notice the conditional search space:

- `gamma` is relevant for RBF and polynomial kernels,
- `degree` is relevant only for the polynomial kernel.

In [ ]:
def make_svc_objective(X_data, y_data, cv_strategy):
    def objective(trial):
        kernel = trial.suggest_categorical(
            "kernel",
            ["linear", "rbf", "poly"],
        )

        C = trial.suggest_float(
            "C",
            1e-3,
            1e3,
            log=True,
        )

        svc_kwargs = {
            "kernel": kernel,
            "C": C,
        }

        if kernel in {"rbf", "poly"}:
            svc_kwargs["gamma"] = trial.suggest_float(
                "gamma",
                1e-4,
                1.0,
                log=True,
            )

        if kernel == "poly":
            svc_kwargs["degree"] = trial.suggest_int(
                "degree",
                2,
                4,
            )

        model = Pipeline([
            ("scale", StandardScaler()),
            ("svc", SVC(**svc_kwargs)),
        ])

        scores = cross_validate(
            model,
            X_data,
            y_data,
            cv=cv_strategy,
            scoring="roc_auc",
        )

        return scores["test_score"].mean()

    return objective

### Activity 4 — audit the objective before running it

Answer first:

1. What data enter the objective?
2. Where is scaling fitted?
3. Why is the scaler inside the pipeline?
4. Does the objective ever access `X_test` or `y_test`?
5. Why is `C` searched with `log=True`?
6. Why is `degree` conditional on the polynomial kernel?

## 5. Run the first Optuna study

We use a seeded TPE sampler for reproducibility.

A small classroom budget is intentional: **30 trials**.

More trials can find a better configuration, but computation is not free. Hyperparameter optimisation is always performed under a budget.

In [ ]:
objective = make_svc_objective(
    X_dev,
    y_dev,
    cv,
)

tpe_study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=RANDOM_STATE),
)

tpe_study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=False,
)

print("Best validation ROC-AUC:", round(tpe_study.best_value, 4))
print("Best parameters:")
tpe_study.best_params

### Stop before celebrating `best_params`

Record:

**Best CV ROC-AUC:**  

**Best kernel:**  

**Best `C`:**  

**Other active hyperparameters:**  

Then answer:

1. Is the best `C` close to the lower or upper boundary?
2. Did the best trial use all possible hyperparameters?
3. Would you expand any range? Why or why not?

## 6. Inspect the whole study, not only the winner

In [ ]:
trials_df = tpe_study.trials_dataframe(
    attrs=("number", "value", "params", "state"),
)

display(
    trials_df.sort_values("value", ascending=False).head(10)
)

In [ ]:
history = trials_df.dropna(subset=["value"]).copy()
history["best_so_far"] = history["value"].cummax()

plt.figure(figsize=(9, 4.5))
plt.plot(
    history["number"],
    history["value"],
    "o",
    alpha=0.65,
    label="trial ROC-AUC",
)
plt.plot(
    history["number"],
    history["best_so_far"],
    linewidth=2,
    label="best so far",
)
plt.xlabel("Trial")
plt.ylabel("CV ROC-AUC")
plt.title("Optuna optimisation history")
plt.legend()
plt.show()

### Activity 5 — read an optimisation history

1. Did most improvement happen early or late?
2. Are several configurations nearly tied?
3. If the top five trials differ by only a tiny amount, what does that say about claiming one exact parameter set is uniquely optimal?
4. Would another 100 trials necessarily improve the conclusion enough to justify the runtime?

In [ ]:
importance = get_param_importances(tpe_study)

importance_df = (
    pd.Series(importance, name="importance")
    .rename_axis("hyperparameter")
    .reset_index()
)

display(importance_df)

plt.figure(figsize=(7, 3.8))
plt.barh(
    importance_df["hyperparameter"],
    importance_df["importance"],
)
plt.xlabel("importance")
plt.title("Optuna parameter importance")
plt.xlim(0, 1)
plt.show()


### Activity 6 — interpret importance cautiously

Complete:

> In this study, __________ appears most influential **within the tested search space**.

Why is that last phrase important?

Could a hyperparameter appear unimportant simply because its search range was too narrow?

## 7. TPE versus random search under the same budget

Optuna's default TPE sampler uses information from previous trials to propose promising future values.

Random search samples independently.

We compare both under the **same number of trials**.  
One run is only an illustration — not a definitive benchmark of optimisation algorithms.

In [ ]:
random_study = optuna.create_study(
    direction="maximize",
    sampler=RandomSampler(seed=RANDOM_STATE),
)

random_study.optimize(
    objective,
    n_trials=30,
    show_progress_bar=False,
)

sampler_comparison = pd.DataFrame([
    {
        "sampler": "TPE",
        "best CV ROC-AUC": tpe_study.best_value,
    },
    {
        "sampler": "Random",
        "best CV ROC-AUC": random_study.best_value,
    },
])

sampler_comparison.round(4)

In [ ]:
def best_so_far(study):
    vals = [
        t.value for t in study.trials
        if t.value is not None
    ]
    return np.maximum.accumulate(vals)

plt.figure(figsize=(9, 4.5))
plt.plot(best_so_far(tpe_study), label="TPE")
plt.plot(best_so_far(random_study), label="Random")
plt.xlabel("Completed trial")
plt.ylabel("Best CV ROC-AUC so far")
plt.title("Search progress under the same trial budget")
plt.legend()
plt.show()

### Activity 7 — avoid an overclaim

Write a cautious conclusion comparing the two samplers.

Your sentence must include:

- the trial budget,
- the fact that this is one dataset,
- one reason the comparison could change with another random seed.

**Your conclusion:**

## 8. Guided AI study activity — challenge your search space

Before opening AI, write your own short justification for:

- the `C` range,
- the `gamma` range,
- logarithmic scaling,
- the choice of ROC-AUC.

**My justification:**

Now use this prompt:

```text
Act as a hyperparameter-tuning tutor.
Do not design the Optuna search space for me and do not rewrite my answer.

Ask me exactly three short questions, one at a time:

1. What behaviour of the model justifies each hyperparameter range?
2. Why should any parameter use a logarithmic rather than linear scale?
3. How do I know that the test set has not influenced the search?

Wait for my answer after each question.
```

Then close the chat and record only:

**One useful AI question:**  

**One search-space or protocol statement I revised:**  

**What evidence or principle justified the revision:**

## 9. Refit the selected configuration and audit the test set once

Only now do we use the test set.

We rebuild the SVC from the best TPE trial, fit it on **all development data**, and evaluate once on the held-out test set.

In [ ]:
best_params = tpe_study.best_params.copy()

kernel = best_params["kernel"]
svc_kwargs = {
    "kernel": kernel,
    "C": best_params["C"],
}

if kernel in {"rbf", "poly"}:
    svc_kwargs["gamma"] = best_params["gamma"]

if kernel == "poly":
    svc_kwargs["degree"] = best_params["degree"]

final_model = Pipeline([
    ("scale", StandardScaler()),
    ("svc", SVC(**svc_kwargs)),
])

final_model.fit(X_dev, y_dev)

test_scores = final_model.decision_function(X_test)
test_pred = final_model.predict(X_test)

final_audit = pd.Series({
    "development best CV ROC-AUC": tpe_study.best_value,
    "test ROC-AUC": roc_auc_score(y_test, test_scores),
    "test accuracy": accuracy_score(y_test, test_pred),
    "test recall": recall_score(y_test, test_pred),
})

final_audit.round(4)

In [ ]:
cm = confusion_matrix(y_test, test_pred)

plt.figure(figsize=(4.8, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Final test confusion matrix")
plt.show()

### Verification checkpoint

1. Is test ROC-AUC close to the best development CV estimate?
2. If the test score is lower, does that prove Optuna failed?
3. Why must we resist changing the search after seeing this test result?
4. Which metric in the final audit matters most for a medical screening application, and why might the answer depend on the cost of errors?

## 10. Optional: save the tuned model and its metadata

The original course material also demonstrated model persistence.  
The useful modern habit is to save **both the fitted model and enough metadata to reproduce the selection**.

In [ ]:
import joblib
import json

joblib.dump(
    final_model,
    "exercise09_tuned_svc.joblib",
)

metadata = {
    "best_cv_roc_auc": float(tpe_study.best_value),
    "best_params": tpe_study.best_params,
    "sampler": "TPESampler(seed=42)",
    "n_trials": len(tpe_study.trials),
    "primary_metric": "ROC-AUC",
}

with open("exercise09_tuning_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

metadata

## 11. Assessed task — Hyperparameter tuning memo (2 points)

You have a budget of **25 new Optuna trials**.

Choose **one** modification:

- revise one search-space boundary,
- remove or add one kernel,
- change the primary metric to a defensible alternative,
- compare TPE with another seeded sampler.

Do **not** use the test set while making the choice.

Submit:

1. **Objective:** what are you optimising?
2. **Search space:** parameters, ranges, and why each range is reasonable.
3. **Validation protocol:** how performance is estimated.
4. **Study evidence:** best score plus one history/importance observation.
5. **Diagnostic:** one boundary, near-tie, or instability you noticed.
6. **Decision:** selected configuration or search strategy.
7. **Test discipline:** explicitly state when the test set was used.
8. **Limitation:** one reason another run or dataset could produce a different result.
9. **Optional AI note:** one AI question and the independently verified revision it caused.

### Marking

| Criterion | Points |
|---|---:|
| Search space and validation protocol are defensible | 0.60 |
| Study is interpreted beyond `best_params` | 0.50 |
| Decision follows from evidence | 0.40 |
| Test discipline is correct | 0.30 |
| Limitation / verified AI note is meaningful | 0.20 |

In [ ]:
# Your 25-trial study.
# Change ONLY choices that you can justify in the memo.

def assessed_objective(trial):
    kernel = trial.suggest_categorical(
        "kernel",
        ["linear", "rbf", "poly"],
    )

    C = trial.suggest_float(
        "C",
        1e-3,
        1e3,
        log=True,
    )

    kwargs = {
        "kernel": kernel,
        "C": C,
    }

    if kernel in {"rbf", "poly"}:
        kwargs["gamma"] = trial.suggest_float(
            "gamma",
            1e-4,
            1.0,
            log=True,
        )

    if kernel == "poly":
        kwargs["degree"] = trial.suggest_int(
            "degree",
            2,
            4,
        )

    model = Pipeline([
        ("scale", StandardScaler()),
        ("svc", SVC(**kwargs)),
    ])

    scores = cross_validate(
        model,
        X_dev,
        y_dev,
        cv=cv,
        scoring="roc_auc",
    )

    return scores["test_score"].mean()


assessed_study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=7),
)

assessed_study.optimize(
    assessed_objective,
    n_trials=25,
    show_progress_bar=False,
)

print("Best CV score:", round(assessed_study.best_value, 4))
assessed_study.best_params

### Your tuning memo

**Objective:**  

**Search space and rationale:**  

**Validation protocol:**  

**Best study result:**  

**Study diagnostic:**  

**Decision:**  

**When the test set was used:**  

**Limitation:**  

**Optional verified AI note:**

# End-of-lab self-check

Before finishing, check that you can:

- [ ] distinguish parameters from hyperparameters,
- [ ] explain why tuning needs validation or cross-validation,
- [ ] keep the test set outside model selection,
- [ ] explain the practical roles of SVM `C`, `gamma`, and `kernel`,
- [ ] recognise parameters that should be searched on a log scale,
- [ ] define an Optuna objective and study,
- [ ] use a conditional search space,
- [ ] inspect trial history rather than only `best_params`,
- [ ] identify a suspicious search-space boundary,
- [ ] compare samplers under the same budget,
- [ ] interpret Optuna parameter importance cautiously,
- [ ] refit the selected model on development data,
- [ ] perform one final test audit,
- [ ] defend a tuning decision after evidence-based questions.

Exercise 10 will move from model tuning to neural networks implemented from first principles in NumPy.